# M7 · Probability for Language — companion notebook

> **Play with this.** A *demonstration, not an assessment* — the module's real assessment is its problem set. Here you build the module's objects with your hands: bigram and trigram language models on a small corpus, sampling at temperature, watching sparsity bite, and running the √d variance experiment *properly* — the one M1's notebook previewed with a promise that this module would explain it.

Companion to the **Probability for Language** module of the Mathematical Foundations track at [llmsforsocialscience.net](https://llmsforsocialscience.net/).

In [ ]:
import numpy as np
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

## 1 · A tiny corpus

Real n-gram models train on billions of words; the mechanics are identical on a few hundred. This corpus is deliberately styled like research-methods prose so the samples stay on brand.

In [ ]:
corpus_text = (
    "the results of the survey were surprising . "
    "the results of the pilot were mixed . "
    "the survey was fielded in march . "
    "the survey was fielded online . "
    "respondents answered the survey in march . "
    "the model predicted the responses well . "
    "the model predicted the outcome poorly . "
    "the pilot informed the survey design . "
    "the results were clear and the model was simple . "
    "the outcome of the pilot was inconclusive . "
    "respondents found the survey clear . "
    "the design of the survey was simple . "
    "the model was fitted to the pilot data . "
    "the survey data were noisy . "
    "the results of the model were inconclusive . "
)
tokens = corpus_text.split()
vocab = sorted(set(tokens))
print(f"{len(tokens)} tokens, vocabulary of {len(vocab)}")

## 2 · A bigram model is counting plus the chain rule

Truncate the chain rule's conditioning to one previous word, and MLE reduces to *count and divide* (the module's appendix says why that is the maximum-likelihood answer).

In [ ]:
bigram_counts = defaultdict(Counter)
for w1, w2 in zip(tokens, tokens[1:]):
    bigram_counts[w1][w2] += 1

def bigram_dist(context):
    c = bigram_counts[context]
    total = sum(c.values())
    return {w: n / total for w, n in c.items()}

print('P(next | "survey") =', {w: round(p, 2) for w, p in sorted(bigram_dist("survey").items(), key=lambda kv: -kv[1])})
print('P(next | "the")    =', {w: round(p, 2) for w, p in sorted(bigram_dist("the").items(), key=lambda kv: -kv[1])[:6]}, "…")

The conditional after *"survey"* is nothing like the conditional after *"the"* — the context did real work. That gap between conditionals is what independence (bag-of-words) throws away.

## 3 · Sampling: the chain rule run forwards

Generation is: sample from the conditional, append, condition on the new suffix, repeat. First with the bigram model, then trigram — watch the text get more plausible with one extra word of context.

In [ ]:
trigram_counts = defaultdict(Counter)
for w1, w2, w3 in zip(tokens, tokens[1:], tokens[2:]):
    trigram_counts[(w1, w2)][w3] += 1

def sample_from(dist_counts):
    words = list(dist_counts.keys())
    p = np.array(list(dist_counts.values()), dtype=float)
    p /= p.sum()
    return words[rng.choice(len(words), p=p)]

def generate_bigram(start, n=12):
    out = [start]
    for _ in range(n):
        out.append(sample_from(bigram_counts[out[-1]]))
    return " ".join(out)

def generate_trigram(w1, w2, n=12):
    out = [w1, w2]
    for _ in range(n):
        ctx = (out[-2], out[-1])
        if ctx not in trigram_counts:
            break                      # sparsity bites — see below
        out.append(sample_from(trigram_counts[ctx]))
    return " ".join(out)

print("bigram:  ", generate_bigram("the"))
print("bigram:  ", generate_bigram("the"))
print("trigram: ", generate_trigram("the", "survey"))
print("trigram: ", generate_trigram("the", "results"))

## 4 · Sparsity bites

More context makes each prediction better — and each context rarer. Count how many possible contexts were ever *seen*: this is the wall n-gram models hit, and the estimation problem transformers were built to solve.

In [ ]:
V = len(vocab)
seen_bi = len(bigram_counts)
seen_tri = len(trigram_counts)
print(f"vocabulary V = {V}")
print(f"bigram contexts seen:  {seen_bi:4d} of {V:>6} possible  ({seen_bi/V:.0%})")
print(f"trigram contexts seen: {seen_tri:4d} of {V*V:>6} possible  ({seen_tri/V**2:.1%})")
print(f"\nat V = 50,000 and 10 words of context, possible contexts = 50000**10 = 10^{10*np.log10(50000):.0f}")
print("no corpus will ever cover that — counting cannot scale; learning a function of the context can.")

## 5 · Temperature on a fixed distribution

The widget's experiment, reproduced in four lines: same scores, different sharpness. Entropy (in bits — M8's unit) tracks the flattening.

In [ ]:
logits = np.array([2.4, 1.8, 1.5, 0.9, 0.4, -0.6, -2.2, -3.0])
names = ["surprising", "clear", "mixed", "inconclusive", "significant", "wrong", "delicious", "purple"]

def softmax_T(z, T):
    z = (z - z.max()) / T          # shift invariance = free numerical stability
    e = np.exp(z)
    return e / e.sum()

def entropy_bits(p):
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, T in zip(axes, [0.3, 1.0, 3.0]):
    p = softmax_T(logits, T)
    ax.bar(range(len(p)), p)
    ax.set_title(f"T = {T}   H = {entropy_bits(p):.2f} bits")
    ax.set_xticks(range(len(p)))
    ax.set_xticklabels(names, rotation=60, ha="right", fontsize=7)
plt.tight_layout(); plt.show()

print("uniform ceiling: log2(8) =", np.log2(8), "bits")

Sample actual words at each temperature and the folk description becomes concrete: low T repeats the winner, high T says *purple*.

In [ ]:
for T in [0.3, 1.0, 3.0]:
    p = softmax_T(logits, T)
    draws = [names[rng.choice(len(names), p=p)] for _ in range(10)]
    print(f"T={T:>3}: {' '.join(draws)}")

## 6 · The √d experiment, done properly

M1's notebook ran this and asked you to hold the question; the module has now supplied the machinery. Claim: for independent unit-variance entries, Var(q·k) = d exactly. Check the variance, then watch what *unscaled* scores at large d do to a softmax.

In [ ]:
for d in [16, 256, 4096]:
    q = rng.standard_normal((20000, d))
    k = rng.standard_normal((20000, d))
    dots = np.sum(q * k, axis=1)
    print(f"d = {d:5d}:  Var(q·k) = {dots.var():9.1f}   (theory: {d})   std = {dots.std():6.1f} ≈ √d = {np.sqrt(d):.1f}")

In [ ]:
# One row of attention scores over 8 positions, at d = 4096, scaled vs unscaled
d = 4096
q = rng.standard_normal(d)
K = rng.standard_normal((8, d))
raw = K @ q
scaled = raw / np.sqrt(d)

for name, s in [("unscaled", raw), ("scaled", scaled)]:
    p = np.exp(s - s.max()); p /= p.sum()
    print(f"{name:9s} scores ≈ {np.round(s, 1)}")
    print(f"          softmax  = {np.round(p, 3)}   entropy = {entropy_bits(p):.2f} bits\n")

The unscaled row saturates: one position takes essentially all the mass (entropy near 0 bits — the accidental T→0 regime), and gradients through the rest die. The scaled row keeps the softmax responsive. **The √d is a z-score** — standardisation applied where you cannot see it. M1's question, closed by experiment.

## 7 · MLE is counting (a planted-truth check)

Plant a known conditional distribution, sample a corpus from it, and verify that count-and-divide recovers the truth — the consistency the appendix promises.

In [ ]:
truth = {"surprising": 0.5, "mixed": 0.3, "clear": 0.2}
words = list(truth.keys()); probs = list(truth.values())
for n in [50, 500, 50000]:
    draws = rng.choice(words, size=n, p=probs)
    est = {w: round(np.mean(draws == w), 3) for w in words}
    print(f"n = {n:6d}: MLE = {est}")
print("truth:      ", truth)

---

**Next:** M8 · Information Theory — where the entropy readout you have been watching becomes the loss function: cross-entropy *is* maximum likelihood, measured in bits.